In [ ]:
!pip install open_clip_torch -q

In [ ]:
from __future__ import annotations
import os
import gc
import json
import time
import hashlib
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import open_clip
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

INPUT = Path("/kaggle/input")
TILES_PATH = INPUT / "datasets/edwardsx/geovision-tiles-sit2"
if not TILES_PATH.exists():
    TILES_PATH = INPUT / "geovision-tiles-sit2"
print(f"Tiles: {TILES_PATH}")

OUTPUT = Path("/kaggle/working")
CKPT_DIR = OUTPUT / "checkpoints_v4_group_split"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
BATCH = 64
LORA_RANK = 16
LR = 2e-5
WEIGHT_DECAY = 0.2
WARMUP_EPOCHS = 2

TILE_PX = 64
TILE_PX_CLIP = 224
N_BANDAS_INPUT = 12

IDX_OPTICAS = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11])
BANDAS_OPTICAS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]

CLASES = [
    "contaminacion_alta_NO2",
    "contaminacion_alta_SO2",
    "ozono_anomalo",
    "vegetacion_densa",
    "suelo_urbano",
]

REMOTECLIP_REPO = "chendelong/RemoteCLIP"
REMOTECLIP_FILE = "RemoteCLIP-ViT-B-32.pt"
SPLIT_MODE = "temporal_group"
VAL_FRAC = 0.06
MIN_VAL_PER_CLASS = 40
TARGET_VAL_PER_CLASS = 60
MAX_VAL_TOTAL = 450
VAL_YEAR_MIN = 2021
VAL_YEAR_MAX = 2024
MIN_VAL_DATES = 8
VAL_MGRS_TILE = "T18NUJ"


## LoRA

In [ ]:
class LoRALinear(nn.Module):
    """Wrapper LoRA sobre nn.Linear. W se congela, solo se entrenan A y B."""
    def __init__(self, linear: nn.Linear, rank: int = 16):
        super().__init__()
        self.linear = linear
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False
        d, k = linear.weight.shape
        self.A = nn.Parameter(torch.randn(d, rank) * 0.01)
        self.B = nn.Parameter(torch.zeros(rank, k))

    @property
    def weight(self):
        return self.linear.weight

    @weight.setter
    def weight(self, w):
        self.linear.weight = w

    @property
    def bias(self):
        return self.linear.bias

    @bias.setter
    def bias(self, b):
        self.linear.bias = b

    def forward(self, x):
        return self.linear(x) + F.linear(x, self.A @ self.B)


def aplicar_lora(module, rank=16, nombres_lora=None):
    """Reemplaza nn.Linear por LoRALinear en atencion y MLP."""
    if nombres_lora is None:
        nombres_lora = {"out_proj", "c_fc", "c_proj"}
    for name, child in module.named_children():
        if isinstance(child, nn.Linear) and name in nombres_lora:
            setattr(module, name, LoRALinear(child, rank))
        else:
            aplicar_lora(child, rank, nombres_lora)

In [ ]:
npz = np.load(TILES_PATH / "tiles_train.npz", allow_pickle=False)
tiles_arr = npz["data"]
bands = list(npz["bands"])
meta = pd.read_parquet(TILES_PATH / "tiles_meta.parquet")

print(f"Tiles: {tiles_arr.shape}  dtype={tiles_arr.dtype}")
print(f"Bandas: {bands}")
print(f"Meta: {meta.shape}")

In [ ]:
print(meta["clase"].value_counts().to_string())
print()
print(f"Fechas unicas: {meta['time_s2'].nunique()}")
print(f"Lat: {meta['lat'].min():.4f} a {meta['lat'].max():.4f}")
print(f"Lon: {meta['lon'].min():.4f} a {meta['lon'].max():.4f}")

In [ ]:
TEMPLATES = {
    "contaminacion_alta_NO2": [
        "Urban area with high NO2 concentration ({v:.2e} mol/m2), heavy traffic.",
        "Elevated NO2 ({v:.2e}) detected in high-traffic urban zone.",
        "NO2 pollution ({v:.2e}) from vehicular emissions in city center.",
        "Industrial area with high nitrogen dioxide ({v:.2e}).",
        "Dense traffic corridor with NO2 levels at {v:.2e}.",
    ],
    "contaminacion_alta_SO2": [
        "Industrial plume with high SO2 concentration ({v:.2e} mol/m2).",
        "Elevated sulfur dioxide ({v:.2e}) near industrial facilities.",
        "SO2 pollution ({v:.2e}) from industrial combustion processes.",
        "High SO2 ({v:.2e}) detected downwind of refinery area.",
        "Sulfur dioxide emission ({v:.2e}) in industrial corridor.",
    ],
    "ozono_anomalo": [
        "Anomalous ozone concentration ({v:.2e} mol/m2) during dry season.",
        "Elevated tropospheric O3 ({v:.2e}) under stable atmospheric conditions.",
        "High ozone ({v:.2e}) associated with biomass burning in the region.",
        "O3 anomaly ({v:.2e}) during photochemical pollution episode.",
        "Above-normal ozone levels ({v:.2e}) in the atmospheric boundary layer.",
    ],
    "vegetacion_densa": [
        "Dense vegetation with NDVI {ndvi:.2f}, healthy photosynthetic activity.",
        "Forest or cropland area, NDVI={ndvi:.2f}, high biomass content.",
        "Lush green vegetation, NDVI {ndvi:.2f}, typical of sugarcane crops.",
        "Vegetated area with strong near-infrared reflectance, NDVI={ndvi:.2f}.",
        "Dense canopy cover, NDVI {ndvi:.2f}, low urban signal.",
    ],
    "suelo_urbano": [
        "Urban built-up area, NDVI={ndvi:.2f}, high impervious surface fraction.",
        "Dense urban fabric with sparse vegetation, NDVI={ndvi:.2f}.",
        "City center with buildings and paved surfaces, NDVI={ndvi:.2f}.",
        "Residential or commercial zone, NDVI={ndvi:.2f}, low green cover.",
        "Urbanized area near DAGMA station, NDVI={ndvi:.2f}.",
    ],
}

print(f"Templates por clase: {len(TEMPLATES)}")
for k, v in TEMPLATES.items():
    print(f"  {k}: {len(v)} variantes")

In [ ]:
def generar_texto(row: pd.Series) -> str:
    v = row["no2"] if pd.notna(row["no2"]) else row["so2"] if pd.notna(row["so2"]) else row["o3"] if pd.notna(row["o3"]) else 0
    template = np.random.choice(TEMPLATES[row["clase"]])
    return template.format(v=v, ndvi=row["ndvi"])

meta["texto_en"] = meta.apply(generar_texto, axis=1)
print("Ejemplos:")
for i in range(5):
    print(f"  [{meta['clase'].iloc[i][:15]}...] {meta['texto_en'].iloc[i][:80]}")

## Dataset

In [ ]:
flat = tiles_arr[:, IDX_OPTICAS, :, :].reshape(len(tiles_arr), N_BANDAS_INPUT, -1)
BAND_MEAN = flat.mean(axis=(0, 2)).astype(np.float32)
BAND_STD = flat.std(axis=(0, 2)).astype(np.float32) + 1e-6

s5p_cols = ["no2", "so2", "o3"]
s5p_vals = meta[s5p_cols].values.astype(np.float32)
s5p_vals = np.nan_to_num(s5p_vals, nan=0.0)
S5P_MEAN = s5p_vals.mean(axis=0)
S5P_STD = s5p_vals.std(axis=0) + 1e-6

print(f"BAND_MEAN: {BAND_MEAN[:4].round(1)}")
print(f"BAND_STD:  {BAND_STD[:4].round(1)}")
print(f"S5P_MEAN:  {S5P_MEAN.round(6)}")
print(f"S5P_STD:   {S5P_STD.round(6)}")

In [ ]:
class TilesDataset(Dataset):
    def __init__(self, indices, tiles, meta, idx_opticas):
        self.indices = indices
        self.tiles = tiles
        self.meta = meta.reset_index(drop=True)
        self.idx_opticas = idx_opticas

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        tile = self.tiles[idx][self.idx_opticas].astype(np.float32)
        tile = (tile - BAND_MEAN[:, None, None]) / BAND_STD[:, None, None]
        tile = np.clip(tile, -3.0, 3.0)
        x = torch.from_numpy(tile)
        if np.random.random() > 0.5:
            x = torch.flip(x, dims=[-1])
        k = np.random.choice([0, 1, 2, 3])
        x = torch.rot90(x, k=k, dims=[-2, -1])
        x = F.interpolate(x.unsqueeze(0), size=TILE_PX_CLIP, mode="bilinear", align_corners=False).squeeze(0)
        texto = self.meta.iloc[idx]["texto_en"]
        return x, texto


def collate(batch):
    imgs = torch.stack([b[0] for b in batch], dim=0)
    txts = [b[1] for b in batch]
    return imgs, txts

In [ ]:
def extraer_anio_s2(valor):
    return int(str(valor)[:4])


def extraer_tile_mgrs(valor):
    return str(valor).split("_")[-1]


def resumen_split(meta, indices_train, indices_val):
    train = meta.iloc[indices_train].copy()
    val = meta.iloc[indices_val].copy()
    fechas_train = set(train["time_s2"].astype(str))
    fechas_val = set(val["time_s2"].astype(str))
    compartidas = fechas_train & fechas_val

    print(f"Train: {len(train)} | Val: {len(val)}")
    print(f"Fechas train: {len(fechas_train)} | Fechas val: {len(fechas_val)} | Compartidas: {len(compartidas)}")
    print(f"Años val: {sorted(val['time_s2'].map(extraer_anio_s2).unique().tolist())}")
    print(f"Tiles MGRS val: {sorted(val['time_s2'].map(extraer_tile_mgrs).unique().tolist())}")
    print("\nVal por clase:")
    print(val["clase"].value_counts().reindex(CLASES, fill_value=0).to_string())
    print("\nTrain por clase:")
    print(train["clase"].value_counts().reindex(CLASES, fill_value=0).to_string())

    return {
        "n_train": len(train),
        "n_val": len(val),
        "fechas_train": len(fechas_train),
        "fechas_val": len(fechas_val),
        "fechas_compartidas": len(compartidas),
        "anios_val": sorted(val["time_s2"].map(extraer_anio_s2).unique().tolist()),
        "tiles_mgrs_val": sorted(val["time_s2"].map(extraer_tile_mgrs).unique().tolist()),
        "val_por_clase": val["clase"].value_counts().reindex(CLASES, fill_value=0).to_dict(),
    }


def seleccionar_fechas_validacion(meta):
    meta_split = meta.copy()
    meta_split["anio_s2"] = meta_split["time_s2"].map(extraer_anio_s2)
    meta_split["tile_mgrs"] = meta_split["time_s2"].map(extraer_tile_mgrs)
    meta_split = meta_split[(meta_split["anio_s2"] >= VAL_YEAR_MIN) & (meta_split["anio_s2"] <= VAL_YEAR_MAX)]
    meta_split = meta_split[meta_split["tile_mgrs"] == VAL_MGRS_TILE]

    tabla = meta_split.groupby(["time_s2", "clase"]).size().unstack(fill_value=0).reindex(columns=CLASES, fill_value=0)
    rng = np.random.default_rng(SEED)
    tabla = tabla.iloc[rng.permutation(len(tabla))]

    seleccionadas = []
    acumulado = pd.Series(0, index=CLASES, dtype=int)
    objetivo = pd.Series(TARGET_VAL_PER_CLASS, index=CLASES, dtype=int)

    while ((acumulado < objetivo).any() or len(seleccionadas) < MIN_VAL_DATES) and len(seleccionadas) < len(tabla):
        deficit = (objetivo - acumulado).clip(lower=0)
        mejores = []
        for fecha, conteos in tabla.drop(index=seleccionadas, errors="ignore").iterrows():
            total_actual = int(acumulado.sum())
            total_fecha = int(conteos[CLASES].sum())
            if total_actual + total_fecha > MAX_VAL_TOTAL and len(seleccionadas) >= MIN_VAL_DATES:
                continue
            aporte_deficit = np.minimum(conteos[CLASES], deficit).sum()
            clases_nuevas = int(((acumulado == 0) & (conteos[CLASES] > 0)).sum())
            balance = -float((acumulado + conteos[CLASES]).std())
            bonus_fecha = 20 if len(seleccionadas) < MIN_VAL_DATES else 0
            exceso = max(0, total_actual + total_fecha - MAX_VAL_TOTAL)
            score = aporte_deficit * 5 + clases_nuevas * 10 + bonus_fecha + balance - exceso * 3
            mejores.append((score, aporte_deficit, fecha))
        if not mejores:
            break
        mejores.sort(reverse=True)
        fecha = mejores[0][2]
        seleccionadas.append(fecha)
        acumulado += tabla.loc[fecha, CLASES].astype(int)

    mask_val = meta["time_s2"].isin(seleccionadas).to_numpy()
    indices_val = np.where(mask_val)[0]
    indices_train = np.where(~mask_val)[0]
    return indices_train, indices_val, seleccionadas


def distancia_minima_misma_clase(meta, indices_train, indices_val):
    train = meta.iloc[indices_train][["clase", "lat", "lon"]].copy()
    val = meta.iloc[indices_val][["clase", "lat", "lon"]].copy()
    filas = []
    for clase in CLASES:
        train_c = train[train["clase"] == clase][["lat", "lon"]].to_numpy(dtype=float)
        val_c = val[val["clase"] == clase][["lat", "lon"]].to_numpy(dtype=float)
        if len(train_c) == 0 or len(val_c) == 0:
            continue
        lat1 = np.radians(val_c[:, 0])[:, None]
        lon1 = np.radians(val_c[:, 1])[:, None]
        lat2 = np.radians(train_c[:, 0])[None, :]
        lon2 = np.radians(train_c[:, 1])[None, :]
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
        dist = 6371.0 * 2 * np.arcsin(np.sqrt(a))
        mins = dist.min(axis=1)
        filas.append({
            "clase": clase,
            "n_val": len(val_c),
            "dist_min_p50_km": float(np.percentile(mins, 50)),
            "dist_min_p10_km": float(np.percentile(mins, 10)),
            "dist_min_p90_km": float(np.percentile(mins, 90)),
        })
    return pd.DataFrame(filas)


indices_train, indices_val, fechas_val = seleccionar_fechas_validacion(meta)
split_info = resumen_split(meta, indices_train, indices_val)
print("\nFechas de validación seleccionadas:")
print(pd.Series(fechas_val).sort_values().to_string(index=False))
print("\nDistancia mínima val -> train por misma clase:")
distancias_split = distancia_minima_misma_clase(meta, indices_train, indices_val)
print(distancias_split.to_string(index=False))

if split_info["fechas_compartidas"] != 0:
    raise RuntimeError("El split temporal comparte fechas entre train y val")
if min(split_info["val_por_clase"].values()) < MIN_VAL_PER_CLASS:
    raise RuntimeError("El split temporal dejó una clase con soporte insuficiente")
if split_info["fechas_val"] < MIN_VAL_DATES:
    raise RuntimeError("El split temporal dejó muy pocas fechas de validación")
if min(split_info["anios_val"]) < VAL_YEAR_MIN or max(split_info["anios_val"]) > VAL_YEAR_MAX:
    raise RuntimeError("El split temporal seleccionó años fuera del rango permitido")
if split_info["tiles_mgrs_val"] != [VAL_MGRS_TILE]:
    raise RuntimeError("El split temporal seleccionó tiles MGRS fuera del permitido")

train_ds = TilesDataset(indices_train, tiles_arr, meta, IDX_OPTICAS)
val_ds = TilesDataset(indices_val, tiles_arr, meta, IDX_OPTICAS)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, collate_fn=collate, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)

print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}")


In [ ]:
ckpt_path = hf_hub_download(repo_id=REMOTECLIP_REPO, filename=REMOTECLIP_FILE)
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True)
clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained=None)
clip_model.load_state_dict(ckpt, strict=False)
print(f"Modelo cargado: {REMOTECLIP_FILE} desde {REMOTECLIP_REPO}")
print()
print("Capas lineales en resblocks[0]:")
b0 = clip_model.visual.transformer.resblocks[0]
for n, m in b0.named_modules():
    if isinstance(m, nn.Linear):
        print(f"  {n}: {tuple(m.weight.shape)}")


In [ ]:
for p in clip_model.parameters():
    p.requires_grad = False

aplicar_lora(clip_model.visual.transformer.resblocks[6:], rank=LORA_RANK)
aplicar_lora(clip_model.transformer.resblocks[6:], rank=LORA_RANK)

n_wrapped = sum(1 for m in clip_model.modules() if isinstance(m, LoRALinear))
print(f"Capas envueltas en LoRA: {n_wrapped}")

for name, p in clip_model.named_parameters():
    if any(k in name for k in ["text_projection", "ln_final", "ln_post", "logit_scale"]):
        p.requires_grad = True
    if "visual.proj" in name:
        p.requires_grad = True

params_lora = sum(p.numel() for p in clip_model.parameters() if p.requires_grad)
params_total = sum(p.numel() for p in clip_model.parameters())
print(f"Params entrenables: {params_lora/1e6:.1f}M / {params_total/1e6:.1f}M ({params_lora/params_total*100:.1f}%)")
print()
print("Top-10 capas entrenables:")
cont = [(n, p.numel()) for n, p in clip_model.named_parameters() if p.requires_grad]
cont.sort(key=lambda x: -x[1])
for n, c in cont[:10]:
    print(f"  {c//1000:>6}K  {n}")


In [ ]:
orig_conv = clip_model.visual.conv1
new_conv = nn.Conv2d(N_BANDAS_INPUT, orig_conv.out_channels, orig_conv.kernel_size, stride=orig_conv.stride, bias=False)

with torch.no_grad():
    idx_r = BANDAS_OPTICAS.index("B4")
    idx_g = BANDAS_OPTICAS.index("B3")
    idx_b = BANDAS_OPTICAS.index("B2")
    w_new = new_conv.weight.data
    w_new[:, idx_r] = orig_conv.weight[:, 0]
    w_new[:, idx_g] = orig_conv.weight[:, 1]
    w_new[:, idx_b] = orig_conv.weight[:, 2]
    rgb_mean = orig_conv.weight.mean(dim=1, keepdim=False)
    for b in range(N_BANDAS_INPUT):
        if b not in (idx_r, idx_g, idx_b):
            w_new[:, b] = rgb_mean * (3.0 / N_BANDAS_INPUT)
    new_conv.weight.copy_(w_new)

clip_model.visual.conv1 = new_conv
print(f"conv1 adaptada: {new_conv.in_channels}ch -> {new_conv.out_channels}ch")

In [ ]:
class VisualProj(nn.Module):
    def __init__(self, visual_dim=512, proj_dim=512):
        super().__init__()
        self.proj = nn.Linear(visual_dim, proj_dim)

    def forward(self, vis_feat):
        return self.proj(vis_feat)

fusion = VisualProj().to(DEVICE)
print(f"VisualProj: ViT(512) -> proj(512)")

In [ ]:
clip_model = clip_model.to(DEVICE)
fusion = fusion.to(DEVICE)

optimizer = torch.optim.AdamW(
    [p for p in clip_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print(f"Optimizer: AdamW, LR={LR}, WD={WEIGHT_DECAY}")
print(f"Scheduler: Cosine, T_max={EPOCHS}")

In [ ]:
def info_nce_loss(img_emb, txt_emb, logit_scale):
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = logit_scale * img_emb @ txt_emb.t()
    labels = torch.arange(len(img_emb), device=img_emb.device)
    loss_i = F.cross_entropy(logits, labels)
    loss_t = F.cross_entropy(logits.t(), labels)
    return (loss_i + loss_t) / 2

def encode_imgs(loader):
    embs, labels = [], []
    for imgs, txts_batch in tqdm(loader, desc="Encoding", leave=False):
        imgs = imgs.to(DEVICE)
        with torch.no_grad():
            vis_feat = clip_model.encode_image(imgs)
            fused = fusion(vis_feat)
        embs.append(fused.cpu())
        labels.extend(txts_batch)
    return torch.cat(embs), labels


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.data")

best_loss = float("inf")
best_epoch = 0
patience = 0
history = []

for epoch in range(1, EPOCHS + 1):
    clip_model.train()
    fusion.train()
    train_loss = 0.0

    for imgs, txts in tqdm(train_loader, desc=f"Epoch {epoch}", leave=False):
        imgs = imgs.to(DEVICE)

        vis_feat = clip_model.encode_image(imgs)
        fused = fusion(vis_feat)

        txt_feat = clip_model.encode_text(open_clip.tokenize(txts).to(DEVICE))

        loss = info_nce_loss(fused, txt_feat, clip_model.logit_scale)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    scheduler.step()

    clip_model.eval()
    fusion.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, txts in tqdm(val_loader, desc=f"  Val", leave=False):
            imgs = imgs.to(DEVICE)
            vis_feat = clip_model.encode_image(imgs)
            fused = fusion(vis_feat)
            txt_feat = clip_model.encode_text(open_clip.tokenize(txts).to(DEVICE))
            loss = info_nce_loss(fused, txt_feat, clip_model.logit_scale)
            val_loss += loss.item()
    val_loss /= len(val_loader)

    history.append((epoch, train_loss, val_loss))
    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch:2d} | train={train_loss:.4f} val={val_loss:.4f} | lr={lr_now:.2e}")

    if val_loss < best_loss:
        best_loss = val_loss
        best_epoch = epoch
        patience = 0
        torch.save({"clip": clip_model.state_dict(), "fusion": fusion.state_dict(),
                    "epoch": epoch, "val_loss": val_loss},
                   CKPT_DIR / "clip_finetuned_best.pt")
    else:
        patience += 1
        if patience >= 3:
            print(f"Early stopping en epoch {epoch}")
            break

print()
print(f"Mejor epoch: {best_epoch} con val_loss={best_loss:.4f}")

In [ ]:
ckpt = torch.load(CKPT_DIR / "clip_finetuned_best.pt", map_location="cpu", weights_only=True)
clip_model.load_state_dict(ckpt["clip"])
fusion.load_state_dict(ckpt["fusion"])
clip_model.eval()
fusion.eval()
print(f"Checkpoint cargado: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

In [ ]:
emb_val, txts_val = encode_imgs(val_loader)
emb_train, _ = encode_imgs(train_loader)
y_val = meta.iloc[indices_val]["clase"].values
y_train = meta.iloc[indices_train]["clase"].values
print(f"Embeddings val: {emb_val.shape}")
print(f"Embeddings train: {emb_train.shape}")

In [ ]:
eval_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)
train_loader_eval = DataLoader(train_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)
emb_val, _ = encode_imgs(eval_loader)
emb_train, _ = encode_imgs(train_loader_eval)
y_val = meta.iloc[indices_val]["clase"].values
y_train = meta.iloc[indices_train]["clase"].values

from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(emb_train.numpy(), y_train)
knn_acc = knn.score(emb_val.numpy(), y_val)
print(f"k-NN accuracy (k=5): {knn_acc:.3f}")

In [ ]:
PROTO_EMBS = []
for c in CLASES:
    idxs = meta[meta["clase"] == c].index[:20]
    txts = meta.loc[idxs, "texto_en"].tolist()
    with torch.no_grad():
        embs = F.normalize(clip_model.encode_text(open_clip.tokenize(txts).to(DEVICE)), dim=-1).cpu()
    PROTO_EMBS.append(embs.mean(dim=0))

PROTO_EMBS = torch.stack(PROTO_EMBS)
sims = F.normalize(emb_val, dim=-1) @ PROTO_EMBS.t()
pred_clase = [CLASES[i] for i in sims.argmax(dim=1)]
acc_zs = (np.array(pred_clase) == y_val).mean()
print(f"Zero-shot accuracy (prototipos reales): {acc_zs:.3f} (chance=0.20)")
print()
print(classification_report(y_val, pred_clase, target_names=CLASES, digits=3))


In [ ]:
for k in [1, 5, 10]:
    top = sims.argsort(dim=-1, descending=True)[:, :k]
    correcto = 0
    for i in range(len(y_val)):
        preds_clase = [CLASES[j] for j in top[i]]
        if y_val[i] in preds_clase:
            correcto += 1
    print(f"R@{k}: {correcto/len(y_val):.3f}")

In [ ]:
final_path = CKPT_DIR / "clip_finetuned_final.pt"
torch.save({
    "clip": clip_model.state_dict(),
    "fusion": fusion.state_dict(),
    "history": history,
    "knn_acc": knn_acc,
    "zero_shot_acc": acc_zs,
}, final_path)

with open(final_path, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()
print(f"Checkpoint: {final_path}")
print(f"MD5: {md5}")

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(y_val, pred_clase, labels=CLASES)
print("Matriz de confusion (zero-shot):")
print(f"{'':25s}", end="")
for c in CLASES:
    print(f"{c[:20]:>20s}", end="")
print()
for i, real in enumerate(CLASES):
    print(f"{real[:25]:25s}", end="")
    for j in range(len(CLASES)):
        print(f"{cm[i,j]:>20d}", end="")
    print()
print()

print("Donde se fue el NO2?")
idx_no2 = y_val == "contaminacion_alta_NO2"
pred_no2 = np.array(pred_clase)[idx_no2]
for c in CLASES:
    print(f"  -> {c[:25]}: {(pred_no2 == c).sum()}")
print()

print("Similaridad de textos (cosine):")
sim_textos = PROTO_EMBS @ PROTO_EMBS.t()
for i, c1 in enumerate(CLASES):
    for j, c2 in enumerate(CLASES):
        if i < j:
            print(f"  {c1[:20]} <-> {c2[:20]}: {sim_textos[i,j]:.3f}")

In [ ]:
print("=== VERIFICACION: modelo sin S5P ===\n")
print("Nota: este modelo fue entrenado SIN S5P, asi que no hay data leakage.")
print("La metrica zero-shot ya es solo visual por construccion.\n")

eval_loader2 = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, collate_fn=collate)
emb_final = []
for imgs, txts_batch in tqdm(eval_loader2, desc="Encoding", leave=False):
    imgs = imgs.to(DEVICE)
    with torch.no_grad():
        fused = fusion(clip_model.encode_image(imgs))
    emb_final.append(F.normalize(fused, dim=-1).cpu())
emb_final = torch.cat(emb_final)

sims_final = emb_final @ PROTO_EMBS.t()
pred_final = [CLASES[i] for i in sims_final.argmax(dim=1)]
acc_final = (np.array(pred_final) == y_val).mean()
print(f"Zero-shot accuracy (solo visual): {acc_final:.3f} (chance=0.20)")
print()
print(classification_report(y_val, pred_final, target_names=CLASES, digits=3))

In [ ]:
import shutil
import kagglehub

MODEL_DIR = OUTPUT / "modelo_clip_v4_group_split"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

best = CKPT_DIR / "clip_finetuned_best.pt"
if best.exists():
    shutil.copy(best, MODEL_DIR / "clip_finetuned_best.pt")

with open(MODEL_DIR / "metrics.json", "w") as f:
    json.dump({
        "epochs": EPOCHS,
        "split_mode": SPLIT_MODE,
        "split_info": split_info,
        "fechas_val": [str(x) for x in fechas_val],
        "best_epoch": int(best_epoch) if isinstance(best_epoch, np.integer) else best_epoch,
        "val_loss": float(best_loss),
        "zero_shot_accuracy": float(acc_zs),
        "knn_accuracy": float(knn_acc),
    }, f, indent=2)

with open(best, "rb") as f:
    md5 = hashlib.md5(f.read()).hexdigest()
with open(MODEL_DIR / "checkpoint.md5", "w") as f:
    f.write(md5)

print(f"Archivos en {MODEL_DIR}:")
for p in sorted(MODEL_DIR.iterdir()):
    size = p.stat().st_size / 1024**2
    print(f"  {p.name}: {size:.2f} MB")
print()
print("Subiendo a Kaggle Dataset...")
handle = "edwardsx/geovision-clip-modelo-v4-group-split"
kagglehub.dataset_upload(
    handle,
    str(MODEL_DIR),
    version_notes=f"Re-entreno v4 group split temporal, sin S5P, {EPOCHS} epochs, zero-shot={acc_zs:.3f}",
)
print(f"Dataset creado: https://www.kaggle.com/datasets/{handle}")